In [ ]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd

import numpy as np
from IPython.display import clear_output

from python_utilities.db_connection import DbConnection


In [ ]:
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

In [ ]:
# start and end is included
# start_date = "2026-05-01"
# end_date = "2026-05-06"
start_date = "2026-06-08"
end_date = "2026-07-01"

In [ ]:
lap = analytics_db.sql_to_df(f"""
    SELECT lt.ticket_uuid, lt.egvp_id, la.attachment_id, lap.value
    FROM llm_tickets lt join llm_attachments la on lt.ticket_uuid = la.ticket_uuid
    join llm_attachments_predictions lap on la.attachment_id = lap.attachment_id
    where lt.created_at >= '{start_date}' and lt.created_at <= '{end_date}'
    and lap.type = 'drittauskunft_egvp' and lap.subtype='is_dritt' and lap.value="'True'"
""")

In [ ]:
lap

In [ ]:
lap_sampled = lap.sample(150, random_state=42)

In [ ]:
attch_ids = lap_sampled.attachment_id.unique().tolist()

In [ ]:
attch_ids

In [ ]:
import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')

In [ ]:
from utils.prod_utils import get_data_by_attachment_id
from python_utilities.db_connection import DbConnection
import boto3
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
# Create session with specific profile
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

In [ ]:
download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp"

In [ ]:
from utils.prod_utils import pivot_attachment_predictions

In [ ]:
all_df = []
for a_id, egvp_id in zip(lap_sampled.attachment_id, lap_sampled.egvp_id):
    dataframe = get_data_by_attachment_id(a_id,analytics_db, s3, pdf_download=True, pdf_download_dir=download_dir)
    pivoted_df = pivot_attachment_predictions(dataframe)
    pivoted_df['pdf_path'] = os.path.join(download_dir, f"{a_id}.pdf")
    all_df.append(pivoted_df)


    
# create one dataframe from all pivoted dataframes
final_df = pd.concat(all_df, ignore_index=True)

In [ ]:
final_df.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_streamlit_show.csv", index=False)

In [ ]:
final_df.shape

In [ ]:
final_df

In [ ]:
final_df.columns

In [ ]:
for a_id, egvp_id in zip(lap.attachment_id, lap.egvp_id):
    old_path = os.path.join(download_dir, f"{a_id}.pdf")
    new_path = os.path.join(download_dir, f"{egvp_id}.pdf")
    if os.path.exists(old_path):
        os.rename(old_path, new_path)

## Add cleaned_text to the streamlit review CSV

Fetch the Textract text for each attachment, clean it with `apply_text_cleaning`, and write the `cleaned_text` column back to `dritt_streamlit_show.csv`.

In [ ]:
from utils.intent_recog_utils import apply_text_cleaning

csv_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_streamlit_show.csv"
review_df = pd.read_csv(csv_path)
review_df.shape

In [ ]:
cleaned_text_by_id = {}
for a_id in review_df.attachment_id.unique():
    data = get_data_by_attachment_id(a_id, analytics_db, s3, pdf_download=False, verbose=False)
    if data.empty or "text" not in data.columns:
        cleaned_text_by_id[a_id] = ""
        continue
    raw_text = data["text"].iloc[0]
    cleaned_text_by_id[a_id] = apply_text_cleaning(raw_text or "")
    clear_output(wait=True)
    print(f"Processed {len(cleaned_text_by_id)} / {review_df.attachment_id.nunique()}")

review_df["cleaned_text"] = review_df.attachment_id.map(cleaned_text_by_id)

In [ ]:
review_df.to_csv(csv_path, index=False)
review_df[["attachment_id", "cleaned_text"]].head()

# Get prediction with new model for dritt and va 

In [1]:
# use gpu task table via intent_recognition ORM

import os
import sys
import json
from ast import literal_eval
from datetime import datetime, timedelta
from configparser import RawConfigParser

import pandas as pd

import numpy as np
from IPython.display import clear_output

# make the intent_recognition package importable (src.* modules)
INTENT_RECOG_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/intent_recognition"
if INTENT_RECOG_DIR not in sys.path:
    sys.path.append(INTENT_RECOG_DIR)

from src.db.database_manager import DatabaseManager, session_scope
from src.graph.repository.orm import GPUTask


start_date = "2026-06-16"
end_date = "2026-07-01"

# read the graph DB URI from secret.ini (same source DbConnection used)
_config = RawConfigParser()
_config.read(os.path.expanduser("~/secret.ini"))
GRAPH_DATABASE_URI = _config.get("GRAPH_DB", "GRAPH_DATABASE_URI")

graph_db_manager = DatabaseManager(db_uri=GRAPH_DATABASE_URI)


2026-06-18 10:18:03.443 | INFO     | src.db.database_manager:__init__:17 - Using provided DB URI.


In [6]:
from sqlalchemy import and_, or_

vermogen_check_inside = """The text must contain at least 2 distinct asset inventory category headings from this list:"""
dritt_check_inside = """5. Output false if the text only references, announces, attaches, forwards, bills, requests, or discusses a Drittauskunft without showing either:"""

# JSON path: model_input -> 'processed_prompt' -> 'user' (as text)
prompt_user = GPUTask.model_input["processed_prompt"]["user"].as_string()

with session_scope(graph_db_manager) as session:
    rows = (
        session.query(GPUTask)
        .filter(GPUTask.created_at >= start_date)
        .filter(GPUTask.created_at <= end_date)
        .filter(
            or_(
                and_(
                    GPUTask.model_name == "vermogenverzeichnis_egvp",
                    prompt_user.contains(vermogen_check_inside, autoescape=True),
                ),
                and_(
                    GPUTask.model_name == "drittauskunft_egvp",
                    prompt_user.contains(dritt_check_inside, autoescape=True),
                ),
            )
        )
        .all()
    )
    data = pd.DataFrame([
        {c.name: getattr(row, c.name) for c in GPUTask.__table__.columns}
        for row in rows
    ])

data


,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
0,8cd1b93a-2fde-4dbd-830a-44dd50e795d5,8715cdc7-0e3c-4054-bbb5-ba2c74e46fa8,e3bdf19c-0b89-5259-a24a-55ff15054be6,vermogenverzeichnis_egvp,done,{'text': 'S. Knop Segelfliegerdamm 89 Obergeri...,"{'answer': '{""is_va"": false}', 'answer_reasoni...",{'is_va': False},0,68869103,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-16 14:24:19.552440,2026-06-16 14:39:01.642039
1,dd2a989f-4b6c-4815-adff-5d8a1485d1ce,008918e7-e284-4e67-b85a-34cfa04a165d,2771f934-2732-5c1f-bb0e-dedd47506632,vermogenverzeichnis_egvp,done,{'text': 'Amtsgericht Tuttlingen VOLLSTRECKUNG...,"{'answer': '{""is_va"": false}', 'answer_reasoni...",{'is_va': False},0,68869005,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-16 14:24:08.606363,2026-06-16 14:39:14.599133
2,e142824c-edc9-4c39-9d90-5feb3c6ce5d0,885447fa-8dd2-4e0c-b41a-fff6a48a15f4,983e9dac-d2d5-5b0f-bbd9-a0e4d8c696c3,vermogenverzeichnis_egvp,done,{'text': 'Bundeszentralamt für Steuern POSTANS...,"{'answer': '{""is_va"": false}', 'answer_reasoni...",{'is_va': False},0,69001588,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:25.735408,2026-06-17 15:26:47.880672
3,1154406e-5c88-4ef9-b770-5c4fdc131e31,701a5d12-1a13-4502-a63a-60a09e2dc1a9,476bed9e-a9fd-51b3-8648-057b125be62f,drittauskunft_egvp,done,{'text': 'Amtsgericht Biberach Büroanschrift: ...,"{'answer': '{""is_dritt"":false}', 'answer_reaso...",{'is_dritt': False},0,69001508,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:15.425869,2026-06-17 15:27:11.051152
4,125ab8d6-d6cd-439a-8108-b1d601d3ba3e,dac5c931-cc00-44f0-85aa-2a1305149f93,91c74570-bdee-52c9-9b87-5918b258273a,drittauskunft_egvp,done,{'text': 'Anlage zu GZ: St Il 4 - S 0229a KEVI...,"{'answer': '{""is_dritt"":false}', 'answer_reaso...",{'is_dritt': False},0,69001444,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:08.933438,2026-06-17 15:27:11.601007
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2065,9705b8c8-cf8e-4717-b8a2-59c67908709e,a090268e-8f27-4c5e-b620-7884f5875d12,983e9dac-d2d5-5b0f-bbd9-a0e4d8c696c3,drittauskunft_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_dritt"":false}', 'answer_reaso...",{'is_dritt': False},0,69001589,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:26.209679,2026-06-17 15:27:36.896806
2066,0efe29ef-cfb6-4abd-9cab-202546a5f646,fb7bcdcd-12d4-49c9-a62c-58886055b243,1517a7db-d6eb-5e89-ad91-75b83a6c5ec9,drittauskunft_egvp,done,{'text': 'Die Datenschutzerklärung zur Informa...,"{'answer': '{""is_dritt"":false}', 'answer_reaso...",{'is_dritt': False},0,69001606,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:28.618651,2026-06-17 15:27:37.976948
2067,c4b7bcfb-1d73-4be8-ae25-2dbc8e1b4f3c,0f1895ef-b4f2-4e09-b6ea-fe5617f65e3b,950721c1-0a7e-5e79-9efb-1f6bbd135186,vermogenverzeichnis_egvp,done,{'text': 'Luisenstraße 62 WOLFGANG SCHRICK 477...,"{'answer': '{""is_va"": false}', 'answer_reasoni...",{'is_va': False},0,69001597,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:27.207165,2026-06-17 15:27:09.447628
2068,410fa65b-27d4-43aa-a486-1ff7f7b24549,a97fc8a5-aee2-4ead-9a51-1631189fedef,950721c1-0a7e-5e79-9efb-1f6bbd135186,vermogenverzeichnis_egvp,done,{'text': 'Es wurde übermittelt: Anfrage Melder...,"{'answer': '{""is_va"": false}', 'answer_reasoni...",{'is_va': False},0,69001598,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:27.676172,2026-06-17 15:27:09.973975


In [7]:
data.model_name.value_counts()

model_name
vermogenverzeichnis_egvp    1035
drittauskunft_egvp          1035
Name: count, dtype: int64

In [8]:
data.created_at.min(), data.created_at.max()

(Timestamp('2026-06-16 13:26:09.780241'),
 Timestamp('2026-06-18 07:23:28.584869'))

In [9]:
data.processed_model_output.value_counts()

processed_model_output
{'is_va': False}       1023
{'is_dritt': False}     882
{'is_dritt': True}      153
{'is_va': True}          12
Name: count, dtype: int64

In [12]:
va_pred_true = data[data.processed_model_output == {'is_va': True}]
va_pred_true

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
16,6173fb1f-0478-4e93-9fa4-ef1dd8c064eb,5da3ffe5-a267-45d5-8726-b2ac3276eb6a,f28d8ac8-9d05-59fa-b38c-930c8f9ac8cd,vermogenverzeichnis_egvp,done,{'text': 'Dieses Dokument ist signiert mit Sig...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69001513,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:16.828239,2026-06-17 15:26:55.153478
49,c5ff2a4c-83a2-4a3f-8b2c-c8fb7bd6688d,a090268e-8f27-4c5e-b620-7884f5875d12,983e9dac-d2d5-5b0f-bbd9-a0e4d8c696c3,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69001589,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:26.212435,2026-06-17 15:27:03.910287
272,8e6e0465-638a-452d-bc59-32caf6655495,e22f2baf-683f-4f59-b679-9485b0ec47ca,e7652ad9-7848-5d96-9c1a-5c102a48859f,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d.: GVin L....,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69005431,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 22:22:31.455149,2026-06-17 23:24:14.017122
303,646485ca-a8b3-418f-8cf6-6e456cfc1549,8d348998-b835-48d6-bab9-e0d5b43116a8,32c5f3fa-b1f2-5424-88e6-ff7ad8ba349e,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Gerichts...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68998515,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:31.208542,2026-06-17 13:40:53.285592
659,21e21273-b414-4d56-bd1a-66d6e449d059,a5ad68ad-3be5-4797-94dc-245f78a5481f,3f0f3390-e8a9-5437-8b52-f7fcb1e95306,vermogenverzeichnis_egvp,done,"{'text': 'Marc-Andre Steurer 88167 Gestratz, d...","{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68869968,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 15:25:16.505751,2026-06-16 15:45:45.118360
774,b0b8227b-c053-4594-b0a8-fea2ec890410,6962d4fb-c57a-4791-ba5f-29b2b5de6054,a59d87be-5a6f-5cf6-b8c2-e298af1f53ed,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68870743,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-16 16:23:24.393262,2026-06-16 16:31:11.684668
899,d4c59c52-79f5-40df-a152-df282f363471,59123a42-c167-4d97-bba4-83e654d9cea6,62d2b946-3317-5a4f-9374-4085f6cf2679,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Hauptger...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68872220,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-16 19:22:27.898951,2026-06-16 19:40:21.111062
990,04e37de4-6509-4bba-b758-58801ce67650,f92216a8-156e-41ba-9c3b-a5cdb3b0f430,8fb6f523-8d61-5633-8533-801ade95a5ef,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d.: GV' in ...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68987685,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 07:23:45.031830,2026-06-17 07:43:54.438078
1179,7656db06-2b91-4531-aa89-9a9001ce4d4f,8e530b94-c84c-48d2-8f23-9625f7039661,7bc72b90-620c-52cc-b0eb-c5bcf2ba78c4,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68991360,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 09:24:38.930443,2026-06-17 09:48:42.675475
1610,ed43d92c-4f71-41a5-bb94-59f96a5be5a4,ddf5416b-bf4e-4e49-ba78-8df3af63f5b6,085e4400-ab7d-5d4e-9bc7-009bca3f5b7b,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68997459,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 12:25:44.675233,2026-06-17 12:26:59.147567


In [13]:
dritt_pred_true = data[data.processed_model_output == {'is_dritt': True}]
dritt_pred_true

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
8,dfddaaf2-a670-46a0-959f-0fee764c7dec,b59bd248-6829-446f-8c47-9b9cb1c69e8f,81da0075-49e9-5437-8955-2be93c648ec7,drittauskunft_egvp,done,{'text': 'Anlage zu GZ: St Il 4 S 0229a KEVIZZ...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001456,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:10.317381,2026-06-17 15:27:16.431121
51,f96ea649-db00-4128-83bf-9beb5b62f377,ed9b5c33-90d7-42da-9574-5c5a7415a344,91c74570-bdee-52c9-9b87-5918b258273a,drittauskunft_egvp,done,{'text': 'Czwink Friedrich-Ebert-Straße 6 Ober...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001443,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:08.473643,2026-06-17 15:27:14.849438
58,8be93608-87a3-4fb9-926c-9ec229750e89,6cd7b667-a190-4aa2-bfd3-7ddf89d5a924,476bed9e-a9fd-51b3-8648-057b125be62f,drittauskunft_egvp,done,{'text': 'Bundeszentralamt für Steuern POSTANS...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001509,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:15.884546,2026-06-17 15:27:20.696180
63,ee584c75-0b8b-4792-9ffe-511bd1631d86,654e003b-15f1-4daa-b376-06f5eb974a70,66f7dd37-6b4a-5f9c-ad2a-93e39d533ecb,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieherin Kerstin Ger...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001520,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:17.766307,2026-06-17 15:27:22.389257
66,4fc268cf-3159-4937-bd9e-6822ca0d27eb,d4e334e2-8af2-40ed-b4d9-5ef3152b3471,b47da63a-7449-58f4-9c41-4f9fa3af28f9,drittauskunft_egvp,done,{'text': 'M. Schmid Frauenstraße 31 Gerichtsvo...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001527,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:18.717820,2026-06-17 15:27:23.512223
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2033,01f8eb9d-c4f8-46ee-876d-e1dc660e1fb7,45a808e3-6102-47e1-82cb-e36cdca4f06f,060b610c-8ea1-5463-8b5d-5b06376a41e0,drittauskunft_egvp,done,{'text': 'Anlage zu GZ: St Il 4 - S 0229a KEVI...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68999920,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:24:14.394621,2026-06-17 14:44:59.288877
2037,5a33a624-3916-43c5-aa8b-c06a528f07b7,1f7bebf4-b737-46ba-9dea-e208a7f0c1f7,b4097886-f05b-5c09-b26a-60c49128db38,drittauskunft_egvp,done,{'text': 'Bundeszentralamt für Steuern POSTANS...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69000092,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:25:35.766288,2026-06-17 14:45:08.625676
2055,ba743a94-127c-4a87-a6ec-927ab4aefd23,584efac0-f877-4748-b200-f94eb3eebc90,bd6e45b4-c331-5276-8129-d87eb48c3166,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieher Werderstraße ...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001479,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:12.634659,2026-06-17 15:27:33.300948
2061,5d89f054-67d3-4c36-906b-36be7c1bcdc6,df887e6c-d923-442c-9d19-9d1b154870d7,eecdfae9-1c8e-5a87-a653-58caa0a50de9,drittauskunft_egvp,done,{'text': 'Bitte stets angeben: DR II 556/26 De...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001564,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:22.921940,2026-06-17 15:27:35.202479


In [14]:
sent_df = pd.concat([va_pred_true, dritt_pred_true], ignore_index=True)
sent_df

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
0,6173fb1f-0478-4e93-9fa4-ef1dd8c064eb,5da3ffe5-a267-45d5-8726-b2ac3276eb6a,f28d8ac8-9d05-59fa-b38c-930c8f9ac8cd,vermogenverzeichnis_egvp,done,{'text': 'Dieses Dokument ist signiert mit Sig...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69001513,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:16.828239,2026-06-17 15:26:55.153478
1,c5ff2a4c-83a2-4a3f-8b2c-c8fb7bd6688d,a090268e-8f27-4c5e-b620-7884f5875d12,983e9dac-d2d5-5b0f-bbd9-a0e4d8c696c3,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69001589,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:26.212435,2026-06-17 15:27:03.910287
2,8e6e0465-638a-452d-bc59-32caf6655495,e22f2baf-683f-4f59-b679-9485b0ec47ca,e7652ad9-7848-5d96-9c1a-5c102a48859f,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d.: GVin L....,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69005431,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 22:22:31.455149,2026-06-17 23:24:14.017122
3,646485ca-a8b3-418f-8cf6-6e456cfc1549,8d348998-b835-48d6-bab9-e0d5b43116a8,32c5f3fa-b1f2-5424-88e6-ff7ad8ba349e,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Gerichts...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68998515,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:31.208542,2026-06-17 13:40:53.285592
4,21e21273-b414-4d56-bd1a-66d6e449d059,a5ad68ad-3be5-4797-94dc-245f78a5481f,3f0f3390-e8a9-5437-8b52-f7fcb1e95306,vermogenverzeichnis_egvp,done,"{'text': 'Marc-Andre Steurer 88167 Gestratz, d...","{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68869968,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 15:25:16.505751,2026-06-16 15:45:45.118360
...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,01f8eb9d-c4f8-46ee-876d-e1dc660e1fb7,45a808e3-6102-47e1-82cb-e36cdca4f06f,060b610c-8ea1-5463-8b5d-5b06376a41e0,drittauskunft_egvp,done,{'text': 'Anlage zu GZ: St Il 4 - S 0229a KEVI...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68999920,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:24:14.394621,2026-06-17 14:44:59.288877
161,5a33a624-3916-43c5-aa8b-c06a528f07b7,1f7bebf4-b737-46ba-9dea-e208a7f0c1f7,b4097886-f05b-5c09-b26a-60c49128db38,drittauskunft_egvp,done,{'text': 'Bundeszentralamt für Steuern POSTANS...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69000092,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:25:35.766288,2026-06-17 14:45:08.625676
162,ba743a94-127c-4a87-a6ec-927ab4aefd23,584efac0-f877-4748-b200-f94eb3eebc90,bd6e45b4-c331-5276-8129-d87eb48c3166,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieher Werderstraße ...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001479,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:12.634659,2026-06-17 15:27:33.300948
163,5d89f054-67d3-4c36-906b-36be7c1bcdc6,df887e6c-d923-442c-9d19-9d1b154870d7,eecdfae9-1c8e-5a87-a653-58caa0a50de9,drittauskunft_egvp,done,{'text': 'Bitte stets angeben: DR II 556/26 De...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001564,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:22.921940,2026-06-17 15:27:35.202479


In [15]:

#sent_df.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/sent_df_before_correction.csv", index=False)

In [ ]:
# APPLY CORRECTION ALGO


In [16]:
import copy
import re

from loguru import logger

# ---- regexes (mirrors src.services.attachment_processing.regexes) -----------

# Matches lines that are headings/titles of a Protokoll document.
PROTOKOLL_DOC_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"protokoll"
    r"|verm[oö0]gens\s*auskunfts?\s*protokoll"
    r")\b"
)

# Matches lines that are headings/titles of an Ergebnis document.
ERGEBNIS_DOC_REGEX = re.compile(
    r"(?im)^\s*ergebnis(?:se)?(?:\s+der\s+verm[oö0]gens\s*auskunft)?\b"
)

# Matches typical Drittauskunft cover-letter phrasings.
DRITTAUSKUNFT_COVER_LETTER_REGEX = re.compile(
    r"(?is)(?:"
    r"drittausk[uü]nfte?\s+nach\s+§?\s*802\s*l\s*zpo\s+beim\s+"
    r"bundeszentralamt\s+f[üu]r\s+steuern\s+einzuholen"
    r"|das\s+ergebnis\s+teile\s+ich\s+ihnen\s+unter\s+[üu]bersendung"
    r"|in\s+der\s+anlage\s+das\s+ergebnis\s+der\s+auskunft\s+beim"
    r")"
)

# Matches Drittauskunft result-payload section labels. Use ``findall`` to count.
DRITTAUSKUNFT_SECTIONS_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"name"
    r"|nachname"
    r"|vorname"
    r"|geburtsdatum"
    r"|kontonummer"
    r"|kontoinhaber"
    r"|verf[üu]gungsberechtigte"
    r")\s*:?",
)

# Matches the title/heading of a Vermögensverzeichnis document.
VERMOEGENSVERZEICHNIS_TITLE_REGEX = re.compile(
    r"(?im)^\s*verm[oö0]gens\s*verzeichnis\b"
)

# Matches Vermögensverzeichnis form-field labels. Use ``findall`` to count.
VERMOEGENSVERZEICHNIS_FORM_FIELDS_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"vorname(?:\(n\)|n)?"
    r"|rufname"
    r"|titel"
    r"|fahrzeuge"
    r"|geschlecht"
    r"|geburtsname"
    r"|bargeld"
    r"|wohnungseinrichtung"
    r"|haushaltsw[äa]sche"
    r"|wertpapiere"
    r"|geburtsdatum"
    r"|anschrift"
    r"|familienstand"
    r")\s*:?",
)

# minimum number of drittauskunft result sections required to consider a
# cover-letter document a real drittauskunft
MIN_DRITTAUSKUNFT_SECTIONS = 3
# minimum number of vermogenverzeichnis form fields required to consider a
# document a real vermogenverzeichnis
MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS = 6


# ---- correction functions ---------------------------------------------------
def correct_drittauskunft_prediction(
    processed_model_output: dict | None,
    text: str | None,
    gpu_task_uuid: str | None = None,
) -> dict | None:
    """Precision-targeted correction for drittauskunft prediction.

    Returns a corrected copy of ``processed_model_output`` (input is not mutated).
    """
    if not processed_model_output or not text:
        return processed_model_output
    is_dritt = processed_model_output.get("is_dritt", False)
    if not is_dritt:
        return processed_model_output

    corrected = copy.deepcopy(processed_model_output)

    # Algorithm 1: protokoll without ergebnis -> not drittauskunft
    if PROTOKOLL_DOC_REGEX.search(text):
        if not ERGEBNIS_DOC_REGEX.search(text):
            corrected["is_dritt"] = False
            logger.info(
                "Drittauskunft prediction corrected to False via Algorithm 1 "
                "(Protokoll without Ergebnis): GPUTask[{}]",
                gpu_task_uuid,
            )
            return corrected

    # Algorithm 2: cover letter without ergebnis / too few sections -> not drittauskunft
    if DRITTAUSKUNFT_COVER_LETTER_REGEX.search(text):
        text = DRITTAUSKUNFT_COVER_LETTER_REGEX.sub("", text)
        if (
            not ERGEBNIS_DOC_REGEX.search(text)
            or len(DRITTAUSKUNFT_SECTIONS_REGEX.findall(text)) < MIN_DRITTAUSKUNFT_SECTIONS
        ):
            corrected["is_dritt"] = False
            logger.info(
                "Drittauskunft prediction corrected to False via Algorithm 2 "
                "(cover letter without Ergebnis or fewer than {} sections): GPUTask[{}]",
                MIN_DRITTAUSKUNFT_SECTIONS,
                gpu_task_uuid,
            )
            return corrected

    return corrected


def correct_vermogenverzeichnis_prediction(
    processed_model_output: dict | None,
    text: str | None,
    gpu_task_uuid: str | None = None,
) -> dict | None:
    """Precision-targeted correction for vermogenverzeichnis prediction.

    Returns a corrected copy of ``processed_model_output`` (input is not mutated).
    """
    if not processed_model_output or not text:
        return processed_model_output
    is_va = processed_model_output.get("is_va", False)
    if not is_va:
        return processed_model_output

    corrected = copy.deepcopy(processed_model_output)

    # Algorithm 1: a real vermogenverzeichnis must contain a title and at least
    # MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS form fields
    has_title = bool(VERMOEGENSVERZEICHNIS_TITLE_REGEX.search(text))
    has_enough_fields = (
        len(VERMOEGENSVERZEICHNIS_FORM_FIELDS_REGEX.findall(text))
        >= MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS
    )
    if not (has_title and has_enough_fields):
        corrected["is_va"] = False
        logger.info(
            "Vermogenverzeichnis prediction corrected to False via Algorithm 1 "
            "(missing Vermogenverzeichnis title or fewer than {} form fields): GPUTask[{}]",
            MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS,
            gpu_task_uuid,
        )
        return corrected

    return corrected


In [17]:
from ast import literal_eval


def _as_dict(value):
    """model_input / processed_model_output may be a dict (DB) or a stringified dict (CSV)."""
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return literal_eval(value)
        except (ValueError, SyntaxError):
            return {}
    return {}


def _extract_clean_text(model_input):
    return _as_dict(model_input).get("clean_text") or ""


# text the model actually saw (model_input['clean_text'])
sent_df["text"] = sent_df["model_input"].map(_extract_clean_text)


def _apply_correction(row):
    pmo = _as_dict(row["processed_model_output"])
    if row["model_name"] == "vermogenverzeichnis_egvp":
        corrected = correct_vermogenverzeichnis_prediction(pmo, row["text"], row["gpu_task_uuid"])
        return pd.Series(
            {"corrected_va": bool(corrected.get("is_va", False)) if corrected else False,
             "corrected_dritt": pd.NA}
        )
    if row["model_name"] == "drittauskunft_egvp":
        corrected = correct_drittauskunft_prediction(pmo, row["text"], row["gpu_task_uuid"])
        return pd.Series(
            {"corrected_va": pd.NA,
             "corrected_dritt": bool(corrected.get("is_dritt", False)) if corrected else False}
        )
    return pd.Series({"corrected_va": pd.NA, "corrected_dritt": pd.NA})


sent_df[["corrected_va", "corrected_dritt"]] = sent_df.apply(_apply_correction, axis=1)
sent_df[["model_name", "attachment_id", "processed_model_output", "corrected_va", "corrected_dritt"]]


2026-06-18 10:37:17.959 | INFO     | __main__:correct_vermogenverzeichnis_prediction:150 - Vermogenverzeichnis prediction corrected to False via Algorithm 1 (missing Vermogenverzeichnis title or fewer than 6 form fields): GPUTask[21e21273-b414-4d56-bd1a-66d6e449d059]
2026-06-18 10:37:17.974 | INFO     | __main__:correct_drittauskunft_prediction:98 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[6038c453-4e2a-4cae-a4f8-367b50a40a01]
2026-06-18 10:37:17.982 | INFO     | __main__:correct_drittauskunft_prediction:98 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[88686b7e-e7f1-45b2-bc07-2d646cb03b31]
2026-06-18 10:37:17.997 | INFO     | __main__:correct_drittauskunft_prediction:98 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[aa358993-32be-47ee-8318-71d4a564a99c]
2026-06-18 10:37:18.011 | INFO     | __main__:correct_drittauskunft_pred

,model_name,attachment_id,processed_model_output,corrected_va,corrected_dritt
0,vermogenverzeichnis_egvp,69001513,{'is_va': True},True,<NA>
1,vermogenverzeichnis_egvp,69001589,{'is_va': True},True,<NA>
2,vermogenverzeichnis_egvp,69005431,{'is_va': True},True,<NA>
3,vermogenverzeichnis_egvp,68998515,{'is_va': True},True,<NA>
4,vermogenverzeichnis_egvp,68869968,{'is_va': True},False,<NA>
...,...,...,...,...,...
160,drittauskunft_egvp,68999920,{'is_dritt': True},<NA>,True
161,drittauskunft_egvp,69000092,{'is_dritt': True},<NA>,True
162,drittauskunft_egvp,69001479,{'is_dritt': True},<NA>,True
163,drittauskunft_egvp,69001564,{'is_dritt': True},<NA>,False


In [24]:
# summary: how many predictions the correction algorithm flipped to False
va_mask = sent_df["model_name"] == "vermogenverzeichnis_egvp"
dritt_mask = sent_df["model_name"] == "drittauskunft_egvp"

print("VA rows:", va_mask.sum(), "| flipped to False:", (sent_df.loc[va_mask, "corrected_va"] == False).sum())
print("Dritt rows:", dritt_mask.sum(), "| flipped to False:", (sent_df.loc[dritt_mask, "corrected_dritt"] == False).sum())


VA rows: 12 | flipped to False: 1
Dritt rows: 153 | flipped to False: 6


In [20]:
corrected = sent_df[
    (sent_df["model_name"] == "drittauskunft_egvp") &
    (sent_df["corrected_dritt"] == False) |
    (sent_df["model_name"] == "vermogenverzeichnis_egvp") &
    (sent_df["corrected_va"] == False)
]
corrected

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at,text,corrected_va,corrected_dritt
4,21e21273-b414-4d56-bd1a-66d6e449d059,a5ad68ad-3be5-4797-94dc-245f78a5481f,3f0f3390-e8a9-5437-8b52-f7fcb1e95306,vermogenverzeichnis_egvp,done,"{'text': 'Marc-Andre Steurer 88167 Gestratz, d...","{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68869968,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 15:25:16.505751,2026-06-16 15:45:45.118360,"Marc-Andre Steurer\n88167 Gestratz, den\nOberg...",False,<NA>
40,6038c453-4e2a-4cae-a4f8-367b50a40a01,1e1135df-0b3f-40de-a9eb-a8a7f76163ba,4db045d5-97e8-574c-98d5-ff75bf296919,drittauskunft_egvp,done,{'text': 'Barbara Ziske Die nachstehend gewähl...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68867765,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 13:26:11.722312,2026-06-16 13:47:48.606599,Barbara Ziske\nDie nachstehend gewählten Formu...,<NA>,False
56,88686b7e-e7f1-45b2-bc07-2d646cb03b31,d293668c-b212-4bb5-a9f7-7291d751e025,b7833220-7b36-5869-a5fe-c9b2a22279a2,drittauskunft_egvp,done,"{'text': 'PETER VOLLMERT Stee, 03.06 den 2016 ...","{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69046115,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-18 07:23:28.093146,2026-06-18 07:57:09.125915,"PETER VOLLMERT\nStee, 03.06 den 2016\nObergeri...",<NA>,False
92,aa358993-32be-47ee-8318-71d4a564a99c,09ecb2cc-64be-4428-b7ba-22ce94446241,5b196f19-cb65-5bc5-a044-792277bb984b,drittauskunft_egvp,done,{'text': 'STEPHAN LANDKAMMER Die nachstehend g...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68989504,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 08:25:54.012925,2026-06-17 08:49:21.416005,STEPHAN LANDKAMMER\nDie nachstehend gewählten ...,<NA>,False
140,5bd4480e-d385-4a43-879d-d12177c4ddec,368c2790-f579-4a13-aabd-1cc6a29e3df5,e340c001-5aac-5f5d-a877-83355bf5b0d9,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieherin Astrid Weiß...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998399,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:22:43.425887,2026-06-17 13:41:42.226535,Obergerichtsvollzieherin Astrid Weiß\nLusenstr...,<NA>,False
152,639bb5c8-8bef-4240-98ed-2a19d8aa4010,af64ed29-2a13-4f1a-88ce-bc942712c673,492ad023-84d2-5516-b189-69fe85b4ec3b,drittauskunft_egvp,done,{'text': 'F. Derschka Die nachstehend gewählte...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69000439,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:27:21.315917,2026-06-17 14:44:55.212412,F. Derschka\nDie nachstehend gewählten Formuli...,<NA>,False
163,5d89f054-67d3-4c36-906b-36be7c1bcdc6,df887e6c-d923-442c-9d19-9d1b154870d7,eecdfae9-1c8e-5a87-a653-58caa0a50de9,drittauskunft_egvp,done,{'text': 'Bitte stets angeben: DR II 556/26 De...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,69001564,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-17 15:14:22.921940,2026-06-17 15:27:35.202479,Bitte stets angeben:\nDR II 556/26\nDer Auftra...,<NA>,False


In [34]:
import sys

sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')
from utils.prod_utils import get_data_by_attachment_id


from python_utilities.db_connection import DbConnection
import boto3
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
# Create session with specific profile
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')


INFO [2026-06-18 10:45:29] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


INFO [2026-06-18 10:45:29] - Found credentials in shared credentials file: ~/.aws/credentials


In [26]:
dritt_download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/send_dritt"
va_download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/send_va"
os.makedirs(dritt_download_dir, exist_ok=True)
os.makedirs(va_download_dir, exist_ok=True)


In [28]:
# downlaod va first
va_true = sent_df[
    (sent_df["model_name"] == "vermogenverzeichnis_egvp") &
    (sent_df["corrected_va"] == True)
]

for a_id in va_true.attachment_id:
    get_data_by_attachment_id(a_id, analytics_db, s3, pdf_download=True, pdf_download_dir="/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/send_va",verbose=False)

In [31]:
dritt_true = sent_df[
    (sent_df["model_name"] == "drittauskunft_egvp") &
    (sent_df["corrected_dritt"] == True)
]
dritt_true.shape

(147, 16)

In [32]:
dritt_true_sample = dritt_true.sample(50, random_state=42)

In [35]:
# downlaod dritt

for a_id in dritt_true_sample.attachment_id:
    get_data_by_attachment_id(a_id, analytics_db, s3, pdf_download=True, pdf_download_dir="/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/send_dritt",verbose=False)

In [66]:
va_false = ["68869968"]
dritt_false = ["68867765","68868888","68869801","68872765"]

In [104]:
va_false_df = data[(data.attachment_id.isin(va_false)) & (data.model_name == "vermogenverzeichnis_egvp")]
va_false_df.reset_index(drop=True, inplace=True)
va_false_df

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
0,21e21273-b414-4d56-bd1a-66d6e449d059,a5ad68ad-3be5-4797-94dc-245f78a5481f,3f0f3390-e8a9-5437-8b52-f7fcb1e95306,vermogenverzeichnis_egvp,done,"{'text': 'Marc-Andre Steurer 88167 Gestratz, d...","{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68869968,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 15:25:16.505751,2026-06-16 15:45:45.118360


In [106]:
check = va_false_df.iloc[0]
print(check.attachment_id)
print(check['model_output']['answer_reasoning_content'])

68869968
Okay, let's tackle this query. The user wants to determine if the provided document text contains the actual "Vermögensverzeichnis" form. The rules are pretty strict, so I need to check each of the three required elements carefully.

First, I need to look for the document header. The text mentions "Vermögensverzeichnis" in a few places. For example, there's a line that says "D. Gerichtsvollzieher(in) errichtet eine Aufstellung mit den nach § 802 C Abs.2 ZPO erforderlichen Angaben als elektronisches Dokument (Vermögensverzeichnis)..." So that's a clear mention of the header. Also, there's "Vermögensverzeichnis" in another part. So that's covered.

Next, the debtor identity. The text has "Marc-Andre Steurer" repeated several times, which is the name. There's also "Fimpel Andre" mentioned, which might be the full name. The address is given as "Gewerbepark Edelweiss Nr. 4, 88138 Weißensberg" and "Altenburg 38A, 88167 Gestratz". So that's the address. The birth date is "02.10.1998"

In [81]:
dritt_false_df = data[(data.attachment_id.isin(dritt_false)) & (data.model_name == "drittauskunft_egvp")]
dritt_false_df.reset_index(drop=True, inplace=True)
dritt_false_df

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
0,70136819-11f1-4c7a-9dfe-4fe82a16b9ae,e484bdba-edc5-4905-b213-48227494e650,86152467-8e68-59b3-9037-844fde1fa431,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieherin Amtsgericht...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68872765,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-16 20:22:17.069756,2026-06-16 20:23:51.135204
1,6038c453-4e2a-4cae-a4f8-367b50a40a01,1e1135df-0b3f-40de-a9eb-a8a7f76163ba,4db045d5-97e8-574c-98d5-ff75bf296919,drittauskunft_egvp,done,{'text': 'Barbara Ziske Die nachstehend gewähl...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68867765,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 13:26:11.722312,2026-06-16 13:47:48.606599
2,cd575998-16d0-4728-b13a-2e885436575f,daf81cd3-67aa-4699-9e79-f4eb2b5b9bd5,982e9e11-0d85-5762-acdf-f88f2c061f89,drittauskunft_egvp,done,{'text': 'Dipl.Rpfl. (FH) J. Kluge 01069 Dresd...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68868888,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-16 14:23:55.837775,2026-06-16 14:39:21.885580
3,3132aeef-ea55-41e7-ab38-b11a7c50e816,3dcfce83-7059-4ebd-96af-f772c151d73d,fea30450-5739-5cfa-8851-ad58075262a2,drittauskunft_egvp,done,{'text': 'AXEL ROMMER Alte Poststraße 66 SPREC...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68869801,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-16 15:24:55.667278,2026-06-16 15:46:23.680694


In [ ]:
check = dritt_false_df.iloc[3]
print(check.attachment_id)
print(check['model_output']['answer_reasoning_content'])

68869801



In [101]:
print(check['model_input']['processed_prompt']['user'])

You are a binary document classification assistant for German debt collection and enforcement documents.

Task:
Decide whether the provided input text itself contains at least one visible actual Drittauskunft result payload or third-party information result.

The input may contain:
- only a Drittauskunft document,
- only non-Drittauskunft documents,
- or a bundle of multiple documents.

Classify true only if BOTH conditions are met:

A. Drittauskunft context is visible.

Context indicators include:
- Drittauskunft / Drittauskünfte / Dritt auskunft
- § 802l ZPO / §802l ZPO / § 802 l ZPO
- Bundeszentralamt für Steuern / Bundeszentralamt fuer Steuern / BZSt
- Kontenabruf / Kontenabrufersuchen / Kontenabrufauskunft
- § 93 AO / § 93b AO, only if connected to Kontenabruf or BZSt

B. A visible result payload is present.

There are two valid result payload types:

TYPE 1: Explicit no-result payload

A no-result payload counts as true even if no bank/account details are present.

Output true if

In [102]:
# print clean text
print(check['model_input']['clean_text'])

Dipl.Rpfl. (FH) J. Kluge
01069 Dresden
Obergerichtsvollzieher
Schnorrstr. 70, IPRO-Gebäude
bei dem Amtsgericht Dresden
Telefon: +49 (0) 351-4799399
Bürozeiten
Fax: +49 (0) 351-20660648
Dienstag 10.00 12.00
Dienstag 13.00 15.00
Bankverbindung:
Dienstkonto: Commerzbank Dresden
IBAN: DE64 8504 0000 0111 3828 00
BIC: COBADEFFXXX
Dipl.RPfl.J.Kluge,OGV;Schnorrstr. 70, 01069 Dresden
Pair Finance GmbH
Knesebeckstraße 62-63
10719 Berlin
Mein Zeichen
Ihr Zeichen
08 DR 599/26
125906745216
Dresden, 16.06.2026
Bitte immer angeben!
Das Büro ist vom 29.06.26-19.07.26 geschlossen!
Zwangsvollstreckungssache
Liquandum Capital GmbH, Knesebeckstraße 62-63, 10719 Berlin
vertr. d.
Pair Finance GmbH, Knesebeckstraße 62-63, 10719 Berlin
gegen
Frau Sarah Hecker, Gohliser Straße 26, 01159 Dresden
Sehr geehrte Damen und Herren,
in oben genannter Sache hat die Schuldnerin im Termin die Vermögensauskunft antragsgemäß abgegeben.
Einen Ausdruck des Vermögensverzeichnisses übersende ich Ihnen zur weiteren Veranlassun